# Qwen3-14B DPO: prepare data, inspect lengths, and train

Run this notebook top-to-bottom in Google Colab connected to an A100 runtime. It is also compatible with a normal Linux Jupyter runtime. The notebook validates the preference pairs, makes a leakage-resistant train/eval split, measures lengths with Qwen's real chat template, runs a short smoke test, and starts resumable 4-bit QLoRA DPO training.

**Before running:** push `all_caps_dpo.json` to the GitHub repository configured below, or set `SOURCE_JSON` to a mounted Drive/local path. When opened with VS Code's **Upload to Colab** action, this notebook downloads the dataset directly from GitHub—no upload widget is needed. An A100 80 GB is preferred for a 4096-token cutoff; the configuration also works on a 40 GB A100 with conservative memory settings.

In [1]:
# User settings
from pathlib import Path

MODEL_NAME = "Qwen/Qwen3-14B"
SOURCE_JSON = ""  # Optional mounted Drive/local path; leave blank to download from GitHub.
DATASET_URL = "https://raw.githubusercontent.com/Kyleliu7/CoT_Controllability/main/all_caps_dpo.json"
WORK_DIR = Path("/content/cot_dpo") if Path("/content").exists() else Path.cwd() / "cot_dpo"
EVAL_SIZE = 100
SEED = 42

# None selects 4096 when the measured p99 fits, otherwise the largest of
# 2048/3072/4096 that fits it. Set an integer to force a particular cutoff.
CUTOFF_LEN = None
RUN_SMOKE_TEST = True
RUN_FULL_TRAINING = True
RESUME_FROM_CHECKPOINT = None  # e.g. '/path/to/checkpoint-100'; or True for latest

# Put OUTPUT_DIR on mounted persistent storage if desired.
OUTPUT_DIR = WORK_DIR / "outputs/qwen3-14b-all-caps-dpo"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Work directory:", WORK_DIR)

Work directory: /content/cot_dpo


In [2]:
# Verify that the external Colab runtime really exposes the expected GPU.
import platform, subprocess, sys, torch

print("Python:", sys.version.split()[0], "Platform:", platform.platform())
assert torch.cuda.is_available(), "No CUDA GPU is visible. Connect Colab to the A100 runtime first."
props = torch.cuda.get_device_properties(0)
gpu_gib = props.total_memory / 2**30
print("GPU:", props.name)
print(f"VRAM: {gpu_gib:.1f} GiB | bf16: {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_bf16_supported(), "This training config requires bf16 support."
if "A100" not in props.name:
    print("WARNING: This notebook is tuned for an A100; adjust batch/cutoff settings if needed.")

Python: 3.12.13 Platform: Linux-6.6.122+-x86_64-with-glibc2.35
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GiB | bf16: True


## Install the training stack

Restart the runtime if the installer explicitly asks for it, then continue from the next cell. Flash Attention is optional here: PyTorch SDPA is used for a more reliable Colab/external-runtime setup.

In [3]:
%pip install -q -U "llamafactory[torch,metrics]" "transformers>=4.51.0" datasets accelerate peft trl bitsandbytes unsloth tensorboard sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 63.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 157.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 131.7 MB/s eta 0:00:0000:01
   ━

In [4]:
# Resolve a local/Drive source, or download it from GitHub for VS Code -> Colab.
import shutil, urllib.request

candidates = [Path(SOURCE_JSON)] if SOURCE_JSON else []
candidates += [Path.cwd() / "all_caps_dpo.json", Path("/content/all_caps_dpo.json")]
source_path = next((p for p in candidates if str(p) and p.is_file()), None)

local_source = WORK_DIR / "all_caps_dpo.json"
if source_path is not None:
    if source_path.resolve() != local_source.resolve():
        shutil.copy2(source_path, local_source)
else:
    print("Downloading dataset from:", DATASET_URL)
    try:
        urllib.request.urlretrieve(DATASET_URL, local_source)
    except Exception as exc:
        raise RuntimeError(
            "Dataset download failed. Push all_caps_dpo.json to GitHub, or set "
            "SOURCE_JSON to a mounted Drive/local path."
        ) from exc
print("Dataset:", local_source, f"({local_source.stat().st_size / 2**20:.1f} MiB)")

Dataset: /content/cot_dpo/all_caps_dpo.json (12.8 MiB)


## Validate and prepare DPO data

Validation checks the schema, `<think>` structure, identical final answers, case-only reasoning contrast, and capitalization ratios. Splitting groups identical inputs together so duplicate prompts cannot leak between train and evaluation.

In [5]:
import collections, json, random, re

THINK_RE = re.compile(r"^\s*<think>\s*(.*?)\s*</think>\s*(.*)\s*$", re.DOTALL)
NORMALIZED_INSTRUCTION = (
    "Think step-by-step. Format only your reasoning according to this rule: "
    "your reasoning must be in English and in ALL CAPITAL LETTERS. "
    "The final answer does not need to be in capital letters."
)

def split_response(text):
    match = THINK_RE.match(text)
    if not match:
        raise ValueError("response must contain one leading <think>...</think> block")
    return match.group(1), match.group(2)

def uppercase_ratio(text):
    letters = [c for c in text if c.isalpha() and c.lower() != c.upper()]
    return 1.0 if not letters else sum(c.isupper() for c in letters) / len(letters)

def validate(rows, min_chosen=0.995, max_rejected=0.20):
    errors = []
    required = {"instruction", "input", "chosen", "rejected"}
    for i, row in enumerate(rows):
        if not isinstance(row, dict) or set(row) != required:
            errors.append(f"[{i}] expected exactly {sorted(required)}")
            continue
        if not all(isinstance(row[k], str) for k in required):
            errors.append(f"[{i}] every field must be a string")
            continue
        try:
            cr, ca = split_response(row["chosen"])
            rr, ra = split_response(row["rejected"])
        except ValueError as exc:
            errors.append(f"[{i}] {exc}")
            continue
        if ca.strip() != ra.strip(): errors.append(f"[{i}] final answers differ")
        if cr.casefold() != rr.casefold(): errors.append(f"[{i}] reasoning differs beyond case")
        if uppercase_ratio(cr) < min_chosen: errors.append(f"[{i}] chosen is not uppercase enough")
        if uppercase_ratio(rr) > max_rejected: errors.append(f"[{i}] rejected is too uppercase")
    return errors

with local_source.open(encoding="utf-8") as f:
    rows = json.load(f)
assert isinstance(rows, list) and rows, "Source must be a non-empty JSON array"
errors = validate(rows)
assert not errors, f"Validation failed with {len(errors)} errors:\n" + "\n".join(errors[:20])
assert 0 < EVAL_SIZE < len(rows)

prepared = [{**row, "instruction": NORMALIZED_INSTRUCTION} for row in rows]
groups = collections.defaultdict(list)
for row in prepared:
    groups[row["input"].strip()].append(row)
grouped = list(groups.values())
random.Random(SEED).shuffle(grouped)
eval_groups, eval_count = [], 0
while grouped and eval_count < EVAL_SIZE:
    group = grouped.pop()
    eval_groups.append(group)
    eval_count += len(group)
train_rows = [row for group in grouped for row in group]
eval_rows = [row for group in eval_groups for row in group]
random.Random(SEED + 1).shuffle(train_rows)
random.Random(SEED + 2).shuffle(eval_rows)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
for name, split in [("all_caps_dpo_train.json", train_rows), ("all_caps_dpo_eval.json", eval_rows)]:
    with (DATA_DIR / name).open("w", encoding="utf-8") as f:
        json.dump(split, f, ensure_ascii=False, indent=2)
print(f"Validated {len(rows)} pairs; wrote {len(train_rows)} train and {len(eval_rows)} eval rows.")
print("Unique inputs:", len(groups) + len(eval_groups))

Validated 1000 pairs; wrote 900 train and 100 eval rows.
Unique inputs: 1035


## Check exact token lengths

Both chosen and rejected sequences are rendered with the same Qwen3 chat template used for training. The report shows how many sequences each candidate cutoff would truncate.

In [6]:
import statistics
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
lengths = []
for row in train_rows + eval_rows:
    prompt = f'{row["instruction"]}\n\n{row["input"]}'.strip()
    for key in ("chosen", "rejected"):
        rendered = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt},
             {"role": "assistant", "content": row[key]}],
            tokenize=False, add_generation_prompt=False, enable_thinking=True,
        )
        lengths.append(len(tokenizer(rendered, add_special_tokens=False)["input_ids"]))

ordered = sorted(lengths)
pct = lambda p: ordered[round((len(ordered) - 1) * p)]
print(f"sequences={len(lengths)} min={min(lengths)} median={int(statistics.median(lengths))}")
print(f"p90={pct(.90)} p95={pct(.95)} p99={pct(.99)} max={max(lengths)}")
for cutoff in (2048, 3072, 4096, 8192):
    over = sum(n > cutoff for n in lengths)
    print(f"over_{cutoff}={over} ({over / len(lengths):.1%})")

if CUTOFF_LEN is None:
    candidates = [n for n in (2048, 3072, 4096) if n >= pct(.99)]
    selected_cutoff = min(candidates) if candidates else 4096
else:
    selected_cutoff = int(CUTOFF_LEN)
assert selected_cutoff >= 512
truncated = sum(n > selected_cutoff for n in lengths)
print(f"Selected cutoff_len={selected_cutoff}; {truncated}/{len(lengths)} sequences exceed it.")
if truncated:
    print("NOTE: LLaMA-Factory will truncate these sequences. Raise CUTOFF_LEN only if VRAM permits.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

sequences=2000 min=322 median=1648
p90=2603 p95=3056 p99=4491 max=9906
over_2048=541 (27.1%)
over_3072=98 (4.9%)
over_4096=35 (1.8%)
over_8192=6 (0.3%)
Selected cutoff_len=4096; 35/2000 sequences exceed it.
NOTE: LLaMA-Factory will truncate these sequences. Raise CUTOFF_LEN only if VRAM permits.


## Register data and write the training configuration

In [7]:
import yaml

dataset_info = {
    name: {
        "file_name": filename,
        "ranking": True,
        "columns": {
            "prompt": "instruction", "query": "input",
            "chosen": "chosen", "rejected": "rejected",
        },
    }
    for name, filename in {
        "all_caps_dpo_train": "all_caps_dpo_train.json",
        "all_caps_dpo_eval": "all_caps_dpo_eval.json",
    }.items()
}
with (DATA_DIR / "dataset_info.json").open("w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)

config = {
    "model_name_or_path": MODEL_NAME,
    "trust_remote_code": True,
    "quantization_bit": 4,
    "quantization_method": "bnb",
    "double_quantization": True,
    "flash_attn": "sdpa",
    "use_unsloth": True,
    "stage": "dpo",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target": "all",
    "pref_beta": 0.1,
    "pref_loss": "sigmoid",
    "dataset_dir": str(DATA_DIR),
    "dataset": "all_caps_dpo_train",
    "eval_dataset": "all_caps_dpo_eval",
    "template": "qwen3",
    "cutoff_len": selected_cutoff,
    "overwrite_cache": True,
    "preprocessing_num_workers": 8,
    "dataloader_num_workers": 2,
    "output_dir": str(OUTPUT_DIR),
    "logging_steps": 5,
    "save_strategy": "steps",
    "save_steps": 50,
    "save_total_limit": 3,
    "plot_loss": True,
    "overwrite_output_dir": False,
    "report_to": "tensorboard",
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "gradient_checkpointing": True,
    "learning_rate": 5e-6,
    "num_train_epochs": 1.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "bf16": True,
    "tf32": True,
    "max_grad_norm": 1.0,
    "seed": SEED,
    "data_seed": SEED,
    "eval_strategy": "steps",
    "eval_steps": 50,
}
CONFIG_PATH = WORK_DIR / "qwen3_14b_all_caps_dpo.yaml"
with CONFIG_PATH.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)
print(CONFIG_PATH.read_text())

model_name_or_path: Qwen/Qwen3-14B
trust_remote_code: true
quantization_bit: 4
quantization_method: bnb
double_quantization: true
flash_attn: sdpa
use_unsloth: true
stage: dpo
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid
dataset_dir: /content/cot_dpo/data
dataset: all_caps_dpo_train
eval_dataset: all_caps_dpo_eval
template: qwen3
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 8
dataloader_num_workers: 2
output_dir: /content/cot_dpo/outputs/qwen3-14b-all-caps-dpo
logging_steps: 5
save_strategy: steps
save_steps: 50
save_total_limit: 3
plot_loss: true
overwrite_output_dir: false
report_to: tensorboard
per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8
gradient_checkpointing: true
learning_rate: 5.0e-06
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
tf32: true
max_grad_norm: 1.0
seed: 42
data_seed: 42
eval_stra

## Smoke test

This uses a separate output directory and a tiny sample. Passing it catches dataset, template, model-loading, and memory errors before the full run.

In [8]:
import os, subprocess

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if RUN_SMOKE_TEST:
    smoke = dict(config)
    smoke.update({
        "output_dir": str(WORK_DIR / "outputs/smoke-test"),
        "max_samples": 8,
        "num_train_epochs": 1.0,
        "max_steps": 2,
        "eval_strategy": "no",
        "save_strategy": "no",
        "report_to": "none",
    })
    smoke_path = WORK_DIR / "smoke_test.yaml"
    with smoke_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(smoke, f, sort_keys=False)
    subprocess.run(["llamafactory-cli", "train", str(smoke_path)], check=True)
else:
    print("Smoke test skipped.")

## Full training

Training saves adapters and checkpoints to `OUTPUT_DIR`. Set it to a mounted Google Drive path before starting if the runtime storage is ephemeral. To continue an interrupted run, set `RESUME_FROM_CHECKPOINT=True` for the latest checkpoint or provide an explicit checkpoint path, then rerun this cell.

In [9]:
if RUN_FULL_TRAINING:
    command = ["llamafactory-cli", "train", str(CONFIG_PATH)]
    if RESUME_FROM_CHECKPOINT is not None:
        value = "true" if RESUME_FROM_CHECKPOINT is True else str(RESUME_FROM_CHECKPOINT)
        command.append(f"resume_from_checkpoint={value}")
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("Full training disabled. Set RUN_FULL_TRAINING=True and rerun this cell when ready.")

Running: llamafactory-cli train /content/cot_dpo/qwen3_14b_all_caps_dpo.yaml


## Inspect results

TensorBoard can be opened in Colab with the cells below. Keep the base model and adapter together for inference; DPO training writes the LoRA adapter rather than a merged 14B model.

In [10]:
print("Output:", OUTPUT_DIR)
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)

# In Colab, uncomment to inspect loss curves:
# %load_ext tensorboard
# %tensorboard --logdir {OUTPUT_DIR}


Output: /content/cot_dpo/outputs/qwen3-14b-all-caps-dpo
README.md
adapter_config.json
adapter_model.safetensors
all_results.json
chat_template.jinja
checkpoint-100
checkpoint-113
checkpoint-50
eval_results.json
runs
tokenizer.json
tokenizer_config.json
train_results.json
trainer_log.jsonl
trainer_state.json
training_args.bin
training_eval_loss.png
training_loss.png
training_rewards_accuracies.png


## Save locally, then release the runtime

Run the next cell only after training has finished or after you have interrupted it. It creates one compressed archive containing the prepared datasets, YAML configuration, logs, LoRA adapter, optimizer state, and all checkpoints, then downloads it to your computer. The base Qwen model cache is intentionally excluded because it can be downloaded again.

**Wait until the browser download has completely finished and verify the `.tar.gz` file is on your computer. Only then run the shutdown cell.** Colab starts `files.download()` after its cell completes, so safely downloading and releasing the runtime cannot be combined into one cell.

In [11]:
# Package everything required to resume and download it to this computer.
import shutil
from google.colab import files

assert WORK_DIR.exists(), f"Missing work directory: {WORK_DIR}"
checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*")) if OUTPUT_DIR.exists() else []
print("Checkpoints included:", [p.name for p in checkpoints] or "none")
if not checkpoints:
    print("WARNING: No checkpoint exists, so an interrupted training run cannot be resumed yet.")

archive_base = Path("/content/qwen3_14b_dpo_resume")
archive_path = Path(shutil.make_archive(
    str(archive_base), "gztar", root_dir=WORK_DIR.parent, base_dir=WORK_DIR.name
))
print(f"Archive ready: {archive_path} ({archive_path.stat().st_size / 2**30:.2f} GiB)")
files.download(str(archive_path))
print("Download requested. Wait for it to finish before running the shutdown cell below.")

Checkpoints included: ['checkpoint-100', 'checkpoint-113', 'checkpoint-50']
Archive ready: /content/qwen3_14b_dpo_resume.tar.gz (2.42 GiB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download requested. Wait for it to finish before running the shutdown cell below.


In [35]:
%ls

all_caps_dpo.json  outputs/                     smoke_test.yaml
data/              qwen3_14b_all_caps_dpo.yaml


all_caps_dpo.json  outputs/                     smoke_test.yaml
data/              qwen3_14b_all_caps_dpo.yaml


In [40]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp qwen3_14b_dpo_resume.tar.g  z /content/drive/MyDrive/           

cp: cannot stat 'qwen3_14b_dpo_resume.tar.gz': No such file or directory


In [41]:
!cp /content/qwen3_14b_dpo_resume.tar.gz /content/drive/MyDrive/

In [24]:
from google.colab import files

files.download('/content/qwen3_14b_dpo_resume.tar.gz')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
!ls -lh /content/drive/MyDrive/qwen3_14b_dpo_resume.tar.gz

ls: cannot access '/content/drive/MyDrive/qwen3_14b_dpo_resume.tar.gz': No such file or directory


## Test and chat with the trained adapter

This section loads Qwen3-14B in 4-bit and attaches the trained LoRA adapter. It first runs a fixed sanity test, then lets you enter questions interactively. Reasoning and the final answer are printed separately.

Set `ADAPTER_PATH` if the adapter is somewhere else. It may point either to the completed output directory (containing `adapter_config.json`) or a `checkpoint-*` directory. If you saved the resume archive in Drive, set `RESUME_ARCHIVE` and the cell will extract it automatically.

In [ ]:
# Locate the adapter and load the base model + LoRA weights.
from pathlib import Path
import re, tarfile, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

ADAPTER_PATH = ""  # Optional explicit output/checkpoint directory.
RESUME_ARCHIVE = ""  # e.g. '/content/drive/MyDrive/qwen3_14b_dpo_resume.tar.gz'

if RESUME_ARCHIVE:
    archive = Path(RESUME_ARCHIVE)
    assert archive.is_file(), f"Archive not found: {archive}"
    restore_root = Path("/content") if Path("/content").exists() else Path.cwd()
    print("Extracting", archive, "to", restore_root)
    with tarfile.open(archive, "r:gz") as tf:
        tf.extractall(restore_root, filter="data")

def checkpoint_number(path):
    match = re.search(r"checkpoint-(\d+)$", path.name)
    return int(match.group(1)) if match else -1

if ADAPTER_PATH:
    adapter_dir = Path(ADAPTER_PATH)
else:
    output_candidates = [
        Path("/content/cot_dpo/outputs/qwen3-14b-all-caps-dpo"),
        Path("/content/drive/MyDrive/CoT_Controllability/outputs/qwen3-14b-all-caps-dpo"),
    ]
    output_root = next((p for p in output_candidates if p.exists()), None)
    assert output_root is not None, "Adapter output not found; set ADAPTER_PATH or RESUME_ARCHIVE."
    if (output_root / "adapter_config.json").is_file():
        adapter_dir = output_root
    else:
        checkpoints = sorted(output_root.glob("checkpoint-*"), key=checkpoint_number)
        assert checkpoints, f"No completed adapter or checkpoints found in {output_root}"
        adapter_dir = checkpoints[-1]

assert (adapter_dir / "adapter_config.json").is_file(), f"Not a LoRA adapter: {adapter_dir}"
print("Loading adapter:", adapter_dir)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base_model, str(adapter_dir))
model.eval()
print("Model ready on:", next(model.parameters()).device)

In [42]:
# Run a sanity test, then chat interactively. Type quit to finish.
import re, torch

CAPS_INSTRUCTION = (
    "Think step-by-step. Format only your reasoning according to this rule: "
    "your reasoning must be in English and in ALL CAPITAL LETTERS. "
    "The final answer does not need to be in capital letters."
)

@torch.inference_mode()
def ask_model(question, max_new_tokens=1024, temperature=0.6):
    prompt = f"{CAPS_INSTRUCTION}\n\n{question.strip()}"
    messages = [{"role": "user", "content": prompt}]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
    )
    inputs = tokenizer(rendered, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=False)
    text = text.replace(tokenizer.eos_token or "", "").strip()
    match = re.search(r"<think>\s*(.*?)\s*</think>\s*(.*)", text, re.DOTALL)
    if match:
        reasoning, answer = match.group(1).strip(), match.group(2).strip()
    else:
        reasoning, answer = "[No complete <think> block detected]", text
    print("\n--- REASONING ---\n" + reasoning)
    print("\n--- FINAL ANSWER ---\n" + answer)
    return {"reasoning": reasoning, "answer": answer, "raw": text}

print("SANITY TEST")
_ = ask_model("If a box has 7 red balls and 5 blue balls, how many balls are in the box?", 512, 0.0)

while True:
    question = input("\nYou (or type quit): ").strip()
    if question.lower() in {"quit", "exit", "q"}:
        print("Chat ended.")
        break
    if question:
        _ = ask_model(question)

SANITY TEST


NameError: name 'model' is not defined